In [1]:
import os
from langchain.chat_models import init_chat_model
llm=init_chat_model("llama-3.1-8b-instant")
llm

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ValueError: Unable to infer model provider for model='llama-3.1-8b-instant', please specify model_provider directly.

In [2]:
import os
from typing import Annotated
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langchain_core.tools import tool

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt

In [ ]:
llm = init_chat_model("llama-3.1-8b-instant")


In [4]:
# Define State
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [5]:
# Build graph
graph_builder = StateGraph(State)


In [6]:
# Human assistance tool
@tool
def human_assistance(query: str) -> str:
    """Request assistance from a human."""
    human_response = interrupt({"query": query})
    return human_response["data"]

In [7]:
# Search tool
search_tool = TavilySearch(max_results=2)

In [8]:
tools = [search_tool, human_assistance]


In [9]:
# Bind tools to LLM
llm_with_tools = llm.bind_tools(tools)

In [10]:
# Chatbot node
def chatbot(state: State):
    message = llm_with_tools.invoke(state["messages"])
    return {"messages": [message]}

In [11]:
# Add nodes
graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)

# Add edges
graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)

In [12]:
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

In [13]:
# Memory
memory = MemorySaver()

In [14]:
# Compile graph
graph = graph_builder.compile(checkpointer=memory)